In [1]:
# ================================================
# Student At-Risk Prediction System
# Step 2 — SQL Database
# ================================================

import sqlite3
import pandas as pd

# Connect to database
conn = sqlite3.connect("student_at_risk.db")

# Helper function to run SQL queries
def run_sql(query):
    return pd.read_sql_query(query, conn)

# Load all 6 CSV files
courses     = pd.read_csv("data/raw/courses.csv")
students    = pd.read_csv("data/raw/students.csv")
assessments = pd.read_csv("data/raw/assessments.csv")
attendance  = pd.read_csv("data/raw/attendance.csv")
vle         = pd.read_csv("data/raw/vle_activity.csv")
support     = pd.read_csv("data/raw/support_sessions.csv")

# Load into database
courses.to_sql("courses",          conn, if_exists="replace", index=False)
students.to_sql("students",        conn, if_exists="replace", index=False)
assessments.to_sql("assessments",  conn, if_exists="replace", index=False)
attendance.to_sql("attendance",    conn, if_exists="replace", index=False)
vle.to_sql("vle_activity",         conn, if_exists="replace", index=False)
support.to_sql("support_sessions", conn, if_exists="replace", index=False)

print("✅ Database created — student_at_risk.db")
print("✅ 6 tables loaded:")
print(f"   courses          → {len(courses)} rows")
print(f"   students         → {len(students)} rows")
print(f"   assessments      → {len(assessments)} rows")
print(f"   attendance       → {len(attendance)} rows")
print(f"   vle_activity     → {len(vle)} rows")
print(f"   support_sessions → {len(support)} rows")
print(f"\n✅ Ready to write SQL queries!")

✅ Database created — student_at_risk.db
✅ 6 tables loaded:
   courses          → 8 rows
   students         → 15000 rows
   assessments      → 55971 rows
   attendance       → 540000 rows
   vle_activity     → 180000 rows
   support_sessions → 15000 rows

✅ Ready to write SQL queries!


In [3]:
##  How many students are in each outcome group?

In [5]:
run_sql("""
    SELECT 
        final_result        AS Outcome,
        COUNT(*)            AS Total_Students,
        ROUND(COUNT(*) * 100.0 / 
            (SELECT COUNT(*) FROM students), 1) AS Percentage
    FROM students
    GROUP BY final_result
    ORDER BY Total_Students DESC
""")

,Outcome,Total_Students,Percentage
0,Pass,6267,41.8
1,Withdrawn,3558,23.7
2,Distinction,2883,19.2
3,Fail,2292,15.3


In [7]:
##Which course has the highest withdrawal rate?

In [9]:
run_sql("""
    SELECT
        c.course_name                                    AS Course,
        c.department                                     AS Department,
        COUNT(s.student_id)                              AS Total_Students,
        SUM(CASE WHEN s.final_result = 'Withdrawn' 
            THEN 1 ELSE 0 END)                           AS Withdrawn,
        ROUND(SUM(CASE WHEN s.final_result = 'Withdrawn' 
            THEN 1.0 ELSE 0 END) / 
            COUNT(s.student_id) * 100, 1)               AS Withdrawal_Rate_Pct
    FROM students s
    JOIN courses c ON s.course_id = c.course_id
    GROUP BY c.course_name
    ORDER BY Withdrawal_Rate_Pct DESC
""")

,Course,Department,Total_Students,Withdrawn,Withdrawal_Rate_Pct
0,Business Studies,Business,1864,460,24.7
1,Health & Social Care,Health,1893,465,24.6
2,Science & Environment,Science,1846,446,24.2
3,Digital Media,Creative,1850,446,24.1
4,Construction,Trades,1884,452,24.0
5,Education & Training,Education,1882,439,23.3
6,Engineering,Engineering,1902,429,22.6
7,Computing & IT,Technology,1879,421,22.4


In [11]:
##Which age group drops out the most?

In [13]:
run_sql("""
    SELECT
        age_band                                         AS Age_Group,
        COUNT(*)                                         AS Total_Students,
        SUM(CASE WHEN final_result = 'Withdrawn' 
            THEN 1 ELSE 0 END)                           AS Withdrawn,
        ROUND(SUM(CASE WHEN final_result = 'Withdrawn' 
            THEN 1.0 ELSE 0 END) / 
            COUNT(*) * 100, 1)                           AS Withdrawal_Rate_Pct,
        ROUND(AVG(CASE WHEN final_result = 'Withdrawn'
            THEN 1.0 ELSE 0 END) * 100, 1)              AS Check_Pct
    FROM students
    GROUP BY age_band
    ORDER BY CASE age_band
        WHEN '0-18'  THEN 1
        WHEN '19-24' THEN 2
        WHEN '25-34' THEN 3
        WHEN '35-54' THEN 4
        WHEN '55+'   THEN 5
    END
""")

,Age_Group,Total_Students,Withdrawn,Withdrawal_Rate_Pct,Check_Pct
0,0-18,4243,1091,25.7,25.7
1,19-24,4793,963,20.1,20.1
2,25-34,3249,794,24.4,24.4
3,35-54,2120,536,25.3,25.3
4,55+,595,174,29.2,29.2


In [15]:
##Do students with disabilities withdraw more?

In [17]:
run_sql("""
    SELECT
        CASE disability
            WHEN 1 THEN 'Has Disability'
            WHEN 0 THEN 'No Disability'
        END                                              AS Disability_Status,
        COUNT(*)                                         AS Total_Students,
        SUM(CASE WHEN final_result = 'Withdrawn'
            THEN 1 ELSE 0 END)                           AS Withdrawn,
        ROUND(SUM(CASE WHEN final_result = 'Withdrawn'
            THEN 1.0 ELSE 0 END) /
            COUNT(*) * 100, 1)                           AS Withdrawal_Rate_Pct,
        ROUND(AVG(CASE WHEN final_result = 'Withdrawn'
            THEN 1.0 ELSE 0 END) * 100, 1)              AS Avg_Withdrawal_Pct
    FROM students
    GROUP BY disability
    ORDER BY disability DESC
""")

,Disability_Status,Total_Students,Withdrawn,Withdrawal_Rate_Pct,Avg_Withdrawal_Pct
0,Has Disability,2234,621,27.8,27.8
1,No Disability,12766,2937,23.0,23.0


In [19]:
##"Students with disabilities showed a notably higher withdrawal rate — this finding directly supports the case for early learning support assessment at enrolment rather than waiting for students to struggle."

In [21]:
## Does deprivation affect withdrawal?

In [23]:
run_sql("""
    SELECT
        imd_band                                         AS Deprivation_Band,
        COUNT(*)                                         AS Total_Students,
        SUM(CASE WHEN final_result = 'Withdrawn'
            THEN 1 ELSE 0 END)                           AS Withdrawn,
        ROUND(SUM(CASE WHEN final_result = 'Withdrawn'
            THEN 1.0 ELSE 0 END) /
            COUNT(*) * 100, 1)                           AS Withdrawal_Rate_Pct
    FROM students
    GROUP BY imd_band
    ORDER BY CASE imd_band
        WHEN '0-10%'   THEN 1
        WHEN '10-20%'  THEN 2
        WHEN '20-30%'  THEN 3
        WHEN '30-40%'  THEN 4
        WHEN '40-50%'  THEN 5
        WHEN '50-60%'  THEN 6
        WHEN '60-70%'  THEN 7
        WHEN '70-80%'  THEN 8
        WHEN '80-90%'  THEN 9
        WHEN '90-100%' THEN 10
    END
""")

,Deprivation_Band,Total_Students,Withdrawn,Withdrawal_Rate_Pct
0,0-10%,1505,467,31.0
1,10-20%,1517,473,31.2
2,20-30%,1485,452,30.4
3,30-40%,1480,305,20.6
4,40-50%,1545,334,21.6
5,50-60%,1473,299,20.3
6,60-70%,1468,297,20.2
7,70-80%,1528,326,21.3
8,80-90%,1467,294,20.0
9,90-100%,1532,311,20.3


In [25]:
##"There was a clear relationship between deprivation and withdrawal — students from the most deprived areas had the highest dropout rates. This tells the college that financial support bursaries and hardship funds are not just nice to have — they directly protect retention rates and ESFA funding."

In [27]:
run_sql("""
    SELECT
        s.student_id,
        s.full_name,
        s.gender,
        s.age_band,
        s.ethnicity,
        s.disability,
        s.region,
        s.highest_education,
        s.imd_band,
        s.num_prev_attempts,
        s.studied_credits,
        s.transport,
        s.part_time_job,
        s.english_first_lang,
        s.final_result,
        c.course_name,
        c.department,
        c.level,
        c.duration_weeks
    FROM students s
    JOIN courses c ON s.course_id = c.course_id
    LIMIT 5
""")

,student_id,full_name,gender,age_band,ethnicity,disability,region,highest_education,imd_band,num_prev_attempts,studied_credits,transport,part_time_job,english_first_lang,final_result,course_name,department,level,duration_weeks
0,STU000001,Leo Hall,Female,25-34,Asian,0,East of England,HE qualification,10-20%,0,120,Walk,0,0,Distinction,Computing & IT,Technology,Level 3,40
1,STU000002,Arjun Taylor,Female,19-24,Mixed,0,Wales,HE qualification,60-70%,0,90,Train,0,0,Pass,Engineering,Engineering,Level 3,40
2,STU000003,Logan Jones,Female,55+,White British,1,South East,A level or equivalent,40-50%,0,120,Walk,1,1,Pass,Business Studies,Business,Level 3,36
3,STU000004,Charlotte Brown,Male,0-18,White British,0,Midlands,Lower than A level,50-60%,0,120,Car,1,0,Pass,Construction,Trades,Level 2,40
4,STU000005,Sophia Smith,Male,19-24,White British,0,Scotland,Lower than A level,90-100%,2,30,Bus,1,0,Withdrawn,Business Studies,Business,Level 3,36


In [29]:
run_sql("""
    SELECT
        s.student_id,
        s.full_name,
        s.gender,
        s.age_band,
        s.ethnicity,
        s.disability,
        s.region,
        s.highest_education,
        s.imd_band,
        s.num_prev_attempts,
        s.studied_credits,
        s.transport,
        s.part_time_job,
        s.english_first_lang,
        s.final_result,
        c.course_name,
        c.department,
        c.level,
        c.duration_weeks,
        COALESCE(ROUND(AVG(a.score), 1), 0)        AS avg_score,
        COALESCE(COUNT(a.assessment_num), 0)        AS total_submitted,
        COALESCE(ROUND(MIN(a.score), 1), 0)        AS lowest_score,
        COALESCE(ROUND(MAX(a.score), 1), 0)        AS highest_score,
        COALESCE(SUM(a.late_submission), 0)         AS late_submissions
    FROM students s
    JOIN courses c      ON s.course_id  = c.course_id
    LEFT JOIN assessments a ON s.student_id = a.student_id
    GROUP BY s.student_id
    LIMIT 5
""")

,student_id,full_name,gender,age_band,ethnicity,disability,region,highest_education,imd_band,num_prev_attempts,...,final_result,course_name,department,level,duration_weeks,avg_score,total_submitted,lowest_score,highest_score,late_submissions
0,STU000001,Leo Hall,Female,25-34,Asian,0,East of England,HE qualification,10-20%,0,...,Distinction,Computing & IT,Technology,Level 3,40,78.1,5,66.4,83.4,0
1,STU000002,Arjun Taylor,Female,19-24,Mixed,0,Wales,HE qualification,60-70%,0,...,Pass,Engineering,Engineering,Level 3,40,55.0,5,43.7,71.5,1
2,STU000003,Logan Jones,Female,55+,White British,1,South East,A level or equivalent,40-50%,0,...,Pass,Business Studies,Business,Level 3,36,56.8,5,52.9,64.5,0
3,STU000004,Charlotte Brown,Male,0-18,White British,0,Midlands,Lower than A level,50-60%,0,...,Pass,Construction,Trades,Level 2,40,62.4,4,50.6,74.3,0
4,STU000005,Sophia Smith,Male,19-24,White British,0,Scotland,Lower than A level,90-100%,2,...,Withdrawn,Business Studies,Business,Level 3,36,39.3,3,18.0,59.6,2


In [31]:
master_table = run_sql("""
    SELECT
        s.student_id,
        s.full_name,
        s.gender,
        s.age_band,
        s.ethnicity,
        s.disability,
        s.region,
        s.highest_education,
        s.imd_band,
        s.num_prev_attempts,
        s.studied_credits,
        s.transport,
        s.part_time_job,
        s.english_first_lang,
        s.final_result,
        c.course_name,
        c.department,
        c.level,
        c.duration_weeks,
        COALESCE(ROUND(AVG(a.score), 1), 0)         AS avg_score,
        COALESCE(COUNT(a.assessment_num), 0)         AS total_submitted,
        COALESCE(ROUND(MIN(a.score), 1), 0)          AS lowest_score,
        COALESCE(ROUND(MAX(a.score), 1), 0)          AS highest_score,
        COALESCE(SUM(a.late_submission), 0)          AS late_submissions,
        COALESCE(ROUND(AVG(att.attendance_pct), 1), 0) AS avg_attendance,
        COALESCE(SUM(v.clicks), 0)                   AS total_clicks,
        COALESCE(ROUND(AVG(v.clicks), 1), 0)         AS avg_weekly_clicks,
        COALESCE(sp.sessions_attended, 0)            AS support_sessions,
        COALESCE(sp.mentor_assigned, 0)              AS mentor_assigned
    FROM students s
    JOIN courses c           ON s.course_id  = c.course_id
    LEFT JOIN assessments a  ON s.student_id = a.student_id
    LEFT JOIN attendance att ON s.student_id = att.student_id
    LEFT JOIN vle_activity v ON s.student_id = v.student_id
    LEFT JOIN support_sessions sp ON s.student_id = sp.student_id
    GROUP BY s.student_id
""")

# Check results
print(f"✅ Master table created!")
print(f"   Rows    : {master_table.shape[0]}")
print(f"   Columns : {master_table.shape[1]}")
print(f"\nColumn names:")
for col in master_table.columns:
    print(f"   → {col}")

# Save to CSV
import os
os.makedirs("data/processed", exist_ok=True)
master_table.to_csv("data/processed/master_table.csv", index=False)
print(f"\n✅ Saved → data/processed/master_table.csv")
print(f"✅ SQL complete — ready for EDA!")

✅ Master table created!
   Rows    : 15000
   Columns : 29

Column names:
   → student_id
   → full_name
   → gender
   → age_band
   → ethnicity
   → disability
   → region
   → highest_education
   → imd_band
   → num_prev_attempts
   → studied_credits
   → transport
   → part_time_job
   → english_first_lang
   → final_result
   → course_name
   → department
   → level
   → duration_weeks
   → avg_score
   → total_submitted
   → lowest_score
   → highest_score
   → late_submissions
   → avg_attendance
   → total_clicks
   → avg_weekly_clicks
   → support_sessions
   → mentor_assigned

✅ Saved → data/processed/master_table.csv
✅ SQL complete — ready for EDA!


In [33]:
# Quick check of the master table
df = master_table.copy()

print("="*50)
print("MASTER TABLE SUMMARY")
print("="*50)

print(f"\nShape : {df.shape[0]} rows × {df.shape[1]} columns")

print(f"\nOutcome breakdown:")
print(df["final_result"].value_counts())

print(f"\nWithdrawal rate : {(df['final_result']=='Withdrawn').mean()*100:.1f}%")

print(f"\nAverage values:")
print(f"   Avg score        : {df['avg_score'].mean():.1f}")
print(f"   Avg attendance   : {df['avg_attendance'].mean():.1f}%")
print(f"   Avg weekly clicks: {df['avg_weekly_clicks'].mean():.1f}")
print(f"   Avg support sess : {df['support_sessions'].mean():.1f}")

print(f"\nFirst 3 rows:")
df.head(3)

MASTER TABLE SUMMARY

Shape : 15000 rows × 29 columns

Outcome breakdown:
final_result
Pass           6267
Withdrawn      3558
Distinction    2883
Fail           2292
Name: count, dtype: int64

Withdrawal rate : 23.7%

Average values:
   Avg score        : 55.1
   Avg attendance   : 71.3%
   Avg weekly clicks: 35.1
   Avg support sess : 3.0

First 3 rows:


,student_id,full_name,gender,age_band,ethnicity,disability,region,highest_education,imd_band,num_prev_attempts,...,avg_score,total_submitted,lowest_score,highest_score,late_submissions,avg_attendance,total_clicks,avg_weekly_clicks,support_sessions,mentor_assigned
0,STU000001,Leo Hall,Female,25-34,Asian,0,East of England,HE qualification,10-20%,0,...,78.1,2160,66.4,83.4,0,91.7,136440,63.2,7,0
1,STU000002,Arjun Taylor,Female,19-24,Mixed,0,Wales,HE qualification,60-70%,0,...,55.0,2160,43.7,71.5,432,78.0,65880,30.5,4,0
2,STU000003,Logan Jones,Female,55+,White British,1,South East,A level or equivalent,40-50%,0,...,56.8,2160,52.9,64.5,0,79.3,52920,24.5,4,1
